# Step 1 : Groupby - Average

In [1]:
# Necessary Libraries
import pandas as pd
import os
from datetime import datetime, time, timedelta
import math
import numpy as np

## Data Awal

### Data Reel

In [2]:
# Pipeline for Data Reel
def merge_reel_data(file_list):
    dataframes = []
    
    for file in file_list:
        if os.path.exists(file):
            df = pd.read_excel(file)
            dataframes.append(df)
            print(f"Berhasil memuat: {file} | Shape: {df.shape}")
        else:
            print(f"GAGAL MEMUAT: {file} tidak ditemukan di direktori.")
            
    if not dataframes:
        raise ValueError("Pipeline dihentikan. Tidak ada satupun file yang valid untuk digabungkan.")
        
    master_reel = pd.concat(dataframes, ignore_index=True)
    
    return master_reel

In [3]:
# Eksekusi Pipeline
daftar_file_reel = [
    "../Data Reel/data reel pm16 0326.xlsx",
    "../Data Reel/data reel pm16 0426.xlsx"
]

reel_pm16 = merge_reel_data(daftar_file_reel)
reel_pm16.head()

Berhasil memuat: ../Data Reel/data reel pm16 0326.xlsx | Shape: (587, 13)
Berhasil memuat: ../Data Reel/data reel pm16 0426.xlsx | Shape: (688, 13)


,Time,Tanggal,Grade,Shift,Reel,Bw,Thickness,MDT,CDT,MDWT,MDS,Brightness,Insp. Status
0,7.36,01.03.26,FC 12,1,26,13.50,0.97,363,135,101,25,86.6,Acc Sotiss
1,8.10,01.03.26,FC 12,1,27,12.44,0.97,436,143,114,24,86.6,Acc Sotiss
2,8.50,01.03.26,FC 12,1,28,12.24,0.93,532,180,123,23,86.4,Acc Sotiss
3,9.30,01.03.26,FC 12,1,29,12.10,0.89,526,182,151,25,86.7,Acc Sotiss
4,10.15,01.03.26,FC 12,1,30,12.06,0.90,573,195,129,22,86.6,Acc Sotiss


### Params PM

In [4]:
# Pipeline for Params PM
def load_and_standardize(file_path):
    _, ext = os.path.splitext(file_path)
    
    if ext.lower() in ['.xlsx', '.xls']:
        df = pd.read_excel(file_path)
    elif ext.lower() == '.csv':
        with open(file_path, encoding='utf-16') as f:
            raw_header = f.readline().strip()
            if raw_header.startswith('"') and '""' in raw_header:
                header = raw_header.strip('"').replace('""', '').split(';')
            else:
                header = [col.strip().strip('"') for col in raw_header.split(';')]
            
            df = pd.read_csv(f, delimiter=';', names=header)
    else:
        raise ValueError(f"Format file tidak dikenali: {ext}")

    # Standarisasi Header
    df.columns = df.columns.str.strip()

    rename_map = {
        'Speed Yankee'   : 'Yankee Speed',
        'Speed Popereel': 'Pope Reel Speed',
        'Pressure Yankee': 'Yankee Pressure',
        'Jet Rasio': 'Jet Wire Ratio',
        'KWH Refiner': 'Load KWH Refiner',
    }
    df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns}, inplace=True)

    # Validasi Waktu sebagai Kunci Utama
    df.dropna(subset=['Time'], inplace=True)
    df['Time'] = pd.to_datetime(df['Time'], dayfirst=True, format='mixed', errors='coerce')

    return df

def build_master_pipeline(file_list):
    dataframes = []
    
    for file in file_list:
        try:
            df = load_and_standardize(file)
            dataframes.append(df)
        except FileNotFoundError:
            print(f"Peringatan: File {file} tidak ditemukan. Dilewati.")
            
    if not dataframes:
        raise ValueError("Tidak ada data yang berhasil dimuat.")

    master_df = pd.concat(dataframes, ignore_index=True)
    master_df.drop_duplicates(subset=['Time'], keep='last', inplace=True)
    master_df.sort_values('Time', inplace=True)
    
    return master_df.reset_index(drop=True)

In [5]:
# Eksekusi Pipeline
file_sources = [
    '../PM Params/Maret-April PM 16.xlsx',
    '../PM Params/28042026_PM16.csv',
    '../PM Params/04052026_PM16.csv',
    '../PM Params/11052026_PM16.csv',
]
raw16 = build_master_pipeline(file_sources)
raw16.info()

<class 'pandas.DataFrame'>
RangeIndex: 32893 entries, 0 to 32892
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Time               32893 non-null  datetime64[us]
 1   Yankee Speed       32893 non-null  float64       
 2   Pope Reel Speed    32893 non-null  float64       
 3   Yankee Pressure    32893 non-null  float64       
 4   Stock Flow         32893 non-null  float64       
 5   Stock Consistency  32893 non-null  float64       
 6   Flow Coating       32893 non-null  float64       
 7   Flow Release       32893 non-null  float64       
 8   Jet Wire Ratio     32893 non-null  float64       
 9   Load KWH Refiner   32893 non-null  float64       
dtypes: datetime64[us](1), float64(9)
memory usage: 2.5 MB


In [6]:
# Create Categories PM Stop/Run
raw16['PM_stop'] = np.where(raw16['Pope Reel Speed'] == 0, "stop", "run")

# Create Coating/(Area.Min) Feature
raw16['Coating/(Area.Min)'] = ((raw16['Flow Coating'] * 60) / (raw16['Yankee Speed'] * 3050))

In [7]:
raw16.info()

<class 'pandas.DataFrame'>
RangeIndex: 32893 entries, 0 to 32892
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Time                32893 non-null  datetime64[us]
 1   Yankee Speed        32893 non-null  float64       
 2   Pope Reel Speed     32893 non-null  float64       
 3   Yankee Pressure     32893 non-null  float64       
 4   Stock Flow          32893 non-null  float64       
 5   Stock Consistency   32893 non-null  float64       
 6   Flow Coating        32893 non-null  float64       
 7   Flow Release        32893 non-null  float64       
 8   Jet Wire Ratio      32893 non-null  float64       
 9   Load KWH Refiner    32893 non-null  float64       
 10  PM_stop             32893 non-null  str           
 11  Coating/(Area.Min)  32442 non-null  float64       
dtypes: datetime64[us](1), float64(10), str(1)
memory usage: 3.1 MB


#### Filter Out

In [8]:
# Filter Out Data
raw16 = raw16[(raw16['Yankee Speed'] >= 500) & (raw16['Pope Reel Speed'] >= 400)]

In [9]:
raw16.info()

<class 'pandas.DataFrame'>
Index: 31488 entries, 0 to 32892
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Time                31488 non-null  datetime64[us]
 1   Yankee Speed        31488 non-null  float64       
 2   Pope Reel Speed     31488 non-null  float64       
 3   Yankee Pressure     31488 non-null  float64       
 4   Stock Flow          31488 non-null  float64       
 5   Stock Consistency   31488 non-null  float64       
 6   Flow Coating        31488 non-null  float64       
 7   Flow Release        31488 non-null  float64       
 8   Jet Wire Ratio      31488 non-null  float64       
 9   Load KWH Refiner    31488 non-null  float64       
 10  PM_stop             31488 non-null  str           
 11  Coating/(Area.Min)  31488 non-null  float64       
dtypes: datetime64[us](1), float64(10), str(1)
memory usage: 3.2 MB


In [10]:
# Delete before-after 0 values in 'Pope Reel Speed'
raw16 = raw16.reset_index(drop=True)
is_zero = np.isclose(raw16['Pope Reel Speed'], 0, atol=1e-5)
mask_to_drop = pd.Series(is_zero).rolling(window=11, center=True, min_periods=1).max().astype(bool)
df_clean = raw16[~mask_to_drop].copy()

In [11]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 31488 entries, 0 to 31487
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Time                31488 non-null  datetime64[us]
 1   Yankee Speed        31488 non-null  float64       
 2   Pope Reel Speed     31488 non-null  float64       
 3   Yankee Pressure     31488 non-null  float64       
 4   Stock Flow          31488 non-null  float64       
 5   Stock Consistency   31488 non-null  float64       
 6   Flow Coating        31488 non-null  float64       
 7   Flow Release        31488 non-null  float64       
 8   Jet Wire Ratio      31488 non-null  float64       
 9   Load KWH Refiner    31488 non-null  float64       
 10  PM_stop             31488 non-null  str           
 11  Coating/(Area.Min)  31488 non-null  float64       
dtypes: datetime64[us](1), float64(10), str(1)
memory usage: 3.0 MB


In [12]:
df_clean.head()

,Time,Yankee Speed,Pope Reel Speed,Yankee Pressure,Stock Flow,Stock Consistency,Flow Coating,Flow Release,Jet Wire Ratio,Load KWH Refiner,PM_stop,Coating/(Area.Min)
0,2026-04-14 04:03:00,1300.43,1085.36,5.51,1228.27,3.20,34.25,34.99,0.92,250.40,run,0.000518
1,2026-04-14 04:04:00,1300.33,1085.45,5.50,1230.08,3.21,34.25,34.99,0.92,250.40,run,0.000518
2,2026-04-14 04:05:00,1299.93,1085.27,5.51,1228.20,3.21,34.25,34.99,0.92,250.39,run,0.000518
3,2026-04-14 04:06:00,1299.63,1085.53,5.49,1230.68,3.21,34.24,34.98,0.92,250.40,run,0.000518
4,2026-04-14 04:07:00,1299.63,1085.45,5.50,1230.56,3.21,34.23,34.98,0.92,250.40,run,0.000518


## Preparation

### Preparation Process

#### Parameter PM

In [13]:
shift1_start = time(7, 0, 1)
shift1_end   = time(15, 0, 0)
shift2_start = time(15, 0, 1)
shift2_end   = time(23, 0, 0)
def assign_shift(t):
    if shift1_start <= t <= shift1_end:
        return 'Shift 1'
    elif shift2_start <= t <= shift2_end:
        return 'Shift 2'
    else:
        return 'Shift 3'

In [14]:
urutan_params_pm = [
    'Date', 'Time', 'Shift', 'Join_Key', 'Timestamp',
    'Creping', 'Yankee Speed', 'Pope Reel Speed',
    'Yankee Pressure', 'Stock Flow', 'Stock Consistency',
    'Flow Coating', 'Flow Release', 'Jet Wire Ratio',
    'Load KWH Refiner','Key_Date', 'PM_stop', 'Coating/(Area.Min)'
]

In [15]:
def preprocess_pm(df_input):
    df = df_input.copy()
    df = df.drop(columns=['Time'])
    df.columns = df.columns.str.strip()
    df.insert(0, 'Creping', (df['Yankee Speed'] - df['Pope Reel Speed']) * 100 / df['Yankee Speed'])
    def extract_time(x):
        if isinstance(x, str):
            return datetime.strptime(x.split(' ')[1], '%H:%M:%S').time()
        else:  # sudah datetime/Timestamp
            return x.time()
    df.insert(0, 'Time', df_input['Time'].apply(extract_time))
    df['Shift'] = df['Time'].apply(assign_shift)
    
    def extract_date(x):
        if isinstance(x, str):
            return x.split(' ')[0]
        else:
            return x.strftime('%d/%m/%y')
    df.insert(0, 'Date', df_input['Time'].apply(extract_date))
    df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%y')
    df['Key_Date'] = [df['Date'][index] - timedelta(days=1) 
                if
                df['Shift'][index] == 'Shift 3' 
                else df['Date'][index] 
                for index in range(len(df['Shift']))]
    df['Join_Key'] = df['Key_Date'].astype(str) + ' ' + df['Time'].astype(str) + ' ' + df['Shift']
    df['Timestamp'] = df['Date'].astype(str) + ' ' + df['Time'].astype(str)
    df['Shift'] = df['Shift'].str.extract(r'(\d+)').astype(int)
    df = df[urutan_params_pm]
    df.drop(columns=['Key_Date'], inplace=True)
    return df

#### Data Reel

In [16]:
def time_to_hms(val):
    s = str(val).strip()
    if s.lower() in {'', 'nan', 'none'}:
        return pd.NA

    # If already contains colon, parse parts directly
    if ':' in s:
        parts = s.split(':')
        h = int(parts[0])
        m = int(parts[1]) if len(parts) > 1 and parts[1] != '' else 0
        sec = int(parts[2]) if len(parts) > 2 and parts[2] != '' else 0

    # If contains dot, treat left as hours and right as minutes (common human shorthand)
    elif '.' in s:
        left, right = s.split('.', 1)
        if int(left) >= 24:
            return pd.NA  # Invalid hour value
        left = 0 if left == "24" else left  # Handle "24" as "00"
        h = int(left) if left != '' else 0

        # If right part is short (1 or 2 digits) treat it as minutes (e.g., "4.1" -> 4:01, "11.55" -> 11:55)
        if len(right) <= 2:
            m = int(right)
            sec = 0
        else:
            # If right part is longer, treat the whole value as a decimal hour (fallback)
            # e.g., "4.125" -> 4.125 hours -> convert fractional hour to minutes
            f = float(s)
            total_minutes = int(round((f - math.floor(f)) * 60))
            m = total_minutes
            sec = 0

    # No separator: treat as hours only (e.g., "6" -> 06:00:00)
    else:
        h = int(float(s))
        m = 0
        sec = 0

    # Normalize minutes >= 60 into hours
    if m >= 60:
        extra_h = m // 60
        h = (h + extra_h) % 24
        m = m % 60

    return f"{h:02d}:{m:02d}:{sec:02d}"

In [17]:
def preprocess_reel(df_input):
    df = df_input.copy()
    df['Time'] = df['Time'].apply(time_to_hms)
    df['Tanggal'] = pd.to_datetime(df['Tanggal'], format='%d.%m.%y')
    df = df.dropna(subset = ['Time']).reset_index(drop = True)
    df['Timestamp'] = df['Tanggal'].astype(str) + ' ' + df['Time'].astype(str)
    df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='%Y-%m-%d %H:%M:%S')
    # Edit Kolom
    df = df.rename(columns={'Tanggal': 'Date'})
    first_cols = ['Date', 'Time', 'Shift', 'Timestamp']
    other_cols = [col for col in df.columns if col not in first_cols]
    df = df[first_cols + other_cols]
    return df

### Apply Preparation

#### Parameter PM

In [18]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 31488 entries, 0 to 31487
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Time                31488 non-null  datetime64[us]
 1   Yankee Speed        31488 non-null  float64       
 2   Pope Reel Speed     31488 non-null  float64       
 3   Yankee Pressure     31488 non-null  float64       
 4   Stock Flow          31488 non-null  float64       
 5   Stock Consistency   31488 non-null  float64       
 6   Flow Coating        31488 non-null  float64       
 7   Flow Release        31488 non-null  float64       
 8   Jet Wire Ratio      31488 non-null  float64       
 9   Load KWH Refiner    31488 non-null  float64       
 10  PM_stop             31488 non-null  str           
 11  Coating/(Area.Min)  31488 non-null  float64       
dtypes: datetime64[us](1), float64(10), str(1)
memory usage: 3.0 MB


In [19]:
pm_16 = preprocess_pm(df_clean)
pm_16.head()

,Date,Time,Shift,Join_Key,Timestamp,Creping,Yankee Speed,Pope Reel Speed,Yankee Pressure,Stock Flow,Stock Consistency,Flow Coating,Flow Release,Jet Wire Ratio,Load KWH Refiner,PM_stop,Coating/(Area.Min)
0,2026-04-14,04:03:00,3,2026-04-13 04:03:00 Shift 3,2026-04-14 04:03:00,16.538376,1300.43,1085.36,5.51,1228.27,3.20,34.25,34.99,0.92,250.40,run,0.000518
1,2026-04-14,04:04:00,3,2026-04-13 04:04:00 Shift 3,2026-04-14 04:04:00,16.525036,1300.33,1085.45,5.50,1230.08,3.21,34.25,34.99,0.92,250.40,run,0.000518
2,2026-04-14,04:05:00,3,2026-04-13 04:05:00 Shift 3,2026-04-14 04:05:00,16.513197,1299.93,1085.27,5.51,1228.20,3.21,34.25,34.99,0.92,250.39,run,0.000518
3,2026-04-14,04:06:00,3,2026-04-13 04:06:00 Shift 3,2026-04-14 04:06:00,16.473920,1299.63,1085.53,5.49,1230.68,3.21,34.24,34.98,0.92,250.40,run,0.000518
4,2026-04-14,04:07:00,3,2026-04-13 04:07:00 Shift 3,2026-04-14 04:07:00,16.480075,1299.63,1085.45,5.50,1230.56,3.21,34.23,34.98,0.92,250.40,run,0.000518


#### Data Reel

In [20]:
reel_pm16 = preprocess_reel(reel_pm16)
reel_pm16.head()

,Date,Time,Shift,Timestamp,Grade,Reel,Bw,Thickness,MDT,CDT,MDWT,MDS,Brightness,Insp. Status
0,2026-03-01,07:36:00,1,2026-03-01 07:36:00,FC 12,26,13.50,0.97,363,135,101,25,86.6,Acc Sotiss
1,2026-03-01,08:01:00,1,2026-03-01 08:01:00,FC 12,27,12.44,0.97,436,143,114,24,86.6,Acc Sotiss
2,2026-03-01,08:05:00,1,2026-03-01 08:05:00,FC 12,28,12.24,0.93,532,180,123,23,86.4,Acc Sotiss
3,2026-03-01,09:03:00,1,2026-03-01 09:03:00,FC 12,29,12.10,0.89,526,182,151,25,86.7,Acc Sotiss
4,2026-03-01,10:15:00,1,2026-03-01 10:15:00,FC 12,30,12.06,0.90,573,195,129,22,86.6,Acc Sotiss


## Pipeline

In [21]:
# 1. MEMBACA DATA
df_reel = reel_pm16.copy()
df_params = pm_16.copy()

# 2. KONVERSI TIMESTAMP
df_reel['Timestamp'] = pd.to_datetime(df_reel['Timestamp'])
df_params['Timestamp'] = pd.to_datetime(df_params['Timestamp'])

# 3. SORT KEY
# REEL: Jam 00-06 ditambah 1 hari (karena di Excel tanggalnya mundur 1 hari dari params)
def create_sort_key_reel(ts):
    if ts.hour < 7:
        return ts + timedelta(days=1)
    return ts

df_reel['Sort_Key'] = df_reel['Timestamp'].apply(create_sort_key_reel)
df_params['Sort_Key'] = df_params['Timestamp']  # Params tidak perlu adjustment

# 4. FILTER BERDASARKAN SORT_KEY RANGE PARAMS
params_sort_min = df_params['Sort_Key'].min()
params_sort_max = df_params['Sort_Key'].max()

df_reel_filtered = df_reel[
    (df_reel['Sort_Key'] >= params_sort_min) & 
    (df_reel['Sort_Key'] <= params_sort_max)
].copy()

df_reel_filtered = df_reel_filtered.sort_values('Sort_Key').reset_index(drop=True)

# 5. DAFTAR VARIABEL
cols_to_avg = [
    'Creping', 'Yankee Speed', 'Pope Reel Speed', 'Yankee Pressure',
    'Stock Flow', 'Stock Consistency', 'Flow Coating', 'Flow Release',
    'Jet Wire Ratio', 'Load KWH Refiner', 'Coating/(Area.Min)'
]

# 6. LOOPING GROUPBY AVERAGE
results = []

for i in range(len(df_reel_filtered) - 1):
    start_time = df_reel_filtered['Timestamp'].iloc[i]
    end_time = df_reel_filtered['Timestamp'].iloc[i + 1]
    start_sort = df_reel_filtered['Sort_Key'].iloc[i]
    end_sort = df_reel_filtered['Sort_Key'].iloc[i + 1]
    
    mask = (df_params['Sort_Key'] >= start_sort) & (df_params['Sort_Key'] < end_sort)
    df_filtered = df_params.loc[mask]
    
    if len(df_filtered) == 0:
        continue
    
    row = {
        'Start_Time': start_time,
        'End_Time': end_time,
        'Data_Count': len(df_filtered)
    }
    
    for col in cols_to_avg:
        mean_val = df_filtered[col].mean()
        row[f'Mean_{col}'] = round(mean_val, 6) if pd.notna(mean_val) else None
    
    results.append(row)

# 7. HASIL
df_result = pd.DataFrame(results)

# 8. SIMPAN
# df_result.to_excel('grouby_params.xlsx', index=False)
print("\n✅ File disimpan: grouby_params.xlsx")


✅ File disimpan: grouby_params.xlsx


# Join Table

## Joining Df_Results and Reel Data

In [22]:
df_groupby = df_result.copy()
df_groupby.head()

,Start_Time,End_Time,Data_Count,Mean_Creping,Mean_Yankee Speed,Mean_Pope Reel Speed,Mean_Yankee Pressure,Mean_Stock Flow,Mean_Stock Consistency,Mean_Flow Coating,Mean_Flow Release,Mean_Jet Wire Ratio,Mean_Load KWH Refiner,Mean_Coating/(Area.Min)
0,2026-04-13 04:05:00,2026-04-13 06:00:00,115,16.499892,1299.988261,1085.491565,5.500522,1227.258783,3.197478,34.243217,34.985130,0.92,245.998087,0.000518
1,2026-04-13 06:00:00,2026-04-14 07:00:00,60,16.498437,1299.973333,1085.498000,5.498333,1220.077500,3.196833,34.242333,34.984333,0.92,246.567167,0.000518
2,2026-04-14 07:00:00,2026-04-14 07:03:00,3,16.498951,1300.163333,1085.650000,5.503333,1220.546667,3.230000,34.246667,34.986667,0.92,242.656667,0.000518
3,2026-04-14 07:03:00,2026-04-14 07:05:00,2,16.490348,1299.730000,1085.400000,5.510000,1222.075000,3.230000,34.235000,34.980000,0.92,242.635000,0.000518
4,2026-04-14 07:05:00,2026-04-14 08:03:00,58,22.535097,1284.496379,995.280517,5.636897,1251.533966,3.195345,32.070517,34.568621,0.92,243.619138,0.000491


In [23]:
df_reel = reel_pm16.copy()
df_reel.head()

,Date,Time,Shift,Timestamp,Grade,Reel,Bw,Thickness,MDT,CDT,MDWT,MDS,Brightness,Insp. Status
0,2026-03-01,07:36:00,1,2026-03-01 07:36:00,FC 12,26,13.50,0.97,363,135,101,25,86.6,Acc Sotiss
1,2026-03-01,08:01:00,1,2026-03-01 08:01:00,FC 12,27,12.44,0.97,436,143,114,24,86.6,Acc Sotiss
2,2026-03-01,08:05:00,1,2026-03-01 08:05:00,FC 12,28,12.24,0.93,532,180,123,23,86.4,Acc Sotiss
3,2026-03-01,09:03:00,1,2026-03-01 09:03:00,FC 12,29,12.10,0.89,526,182,151,25,86.7,Acc Sotiss
4,2026-03-01,10:15:00,1,2026-03-01 10:15:00,FC 12,30,12.06,0.90,573,195,129,22,86.6,Acc Sotiss


In [24]:
# Create Join Key in df_groupby
df_groupby['Join_Key_Timestamp'] = df_groupby['End_Time'] #End_Time
# Create Join Key in df_reel
df_reel['Join_Key_Timestamp'] = df_reel['Timestamp']

In [25]:
# Join df_groupby with df_reel on Join_Key_Timestamp
df_joined = pd.merge(df_groupby, df_reel, left_on='Join_Key_Timestamp', right_on='Join_Key_Timestamp', how='inner')
df_joined.head()

,Start_Time,End_Time,Data_Count,Mean_Creping,Mean_Yankee Speed,Mean_Pope Reel Speed,Mean_Yankee Pressure,Mean_Stock Flow,Mean_Stock Consistency,Mean_Flow Coating,...,Grade,Reel,Bw,Thickness,MDT,CDT,MDWT,MDS,Brightness,Insp. Status
0,2026-04-13 04:05:00,2026-04-13 06:00:00,115,16.499892,1299.988261,1085.491565,5.500522,1227.258783,3.197478,34.243217,...,N 11,24,11.98,0.65,848,563,217,20,87.1,Acc Sotiss
1,2026-04-13 06:00:00,2026-04-14 07:00:00,60,16.498437,1299.973333,1085.498000,5.498333,1220.077500,3.196833,34.242333,...,N 11,25,12.26,0.64,824,567,238,27,87.0,Acc Sotiss
2,2026-04-14 07:00:00,2026-04-14 07:03:00,3,16.498951,1300.163333,1085.650000,5.503333,1220.546667,3.230000,34.246667,...,T 14,SU,13.72,0.71,997,720,108,20,87.2,Acc Sotiss
3,2026-04-14 07:03:00,2026-04-14 07:05:00,2,16.490348,1299.730000,1085.400000,5.510000,1222.075000,3.230000,34.235000,...,N 11,26,12.04,0.64,1394,570,350,30,87.0,Acc Sotiss
4,2026-04-14 07:05:00,2026-04-14 08:03:00,58,22.535097,1284.496379,995.280517,5.636897,1251.533966,3.195345,32.070517,...,T 14,27,14.48,0.93,768,559,250,30,86.0,Acc Sotiss


## Joining df_joined with BB Table

### Data BB

In [26]:
# Pipeline for Data Reel
def merge_BB_data(file_list):
    dataframes = []
    
    for file in file_list:
        if os.path.exists(file):
            df = pd.read_excel(file, engine='openpyxl')
            dataframes.append(df)
            df.drop(columns=['Grade'], inplace=True)
            print(f"Berhasil memuat: {file} | Shape: {df.shape}")
        else:
            print(f"GAGAL MEMUAT: {file} tidak ditemukan di direktori.")
            
    if not dataframes:
        raise ValueError("Pipeline dihentikan. Tidak ada satupun file yang valid untuk digabungkan.")
        
    master_reel = pd.concat(dataframes, ignore_index=True)
    
    return master_reel

In [27]:
# Eksekusi Pipeline
file_sources = [
    "E:\Kuliah\Sun Paper Source\Efficiency\BB_Maret_PM16.xlsx",
    "E:\Kuliah\Sun Paper Source\Efficiency\BB_April_PM16.xlsx"
]
df_BB = merge_BB_data(file_sources)
df_BB.tail()

Berhasil memuat: E:\Kuliah\Sun Paper Source\Efficiency\BB_Maret_PM16.xlsx | Shape: (31, 10)
Berhasil memuat: E:\Kuliah\Sun Paper Source\Efficiency\BB_April_PM16.xlsx | Shape: (31, 10)


<>:3: SyntaxWarning: "\K" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\K"? A raw string is also an option.
<>:4: SyntaxWarning: "\K" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\K"? A raw string is also an option.
<>:3: SyntaxWarning: "\K" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\K"? A raw string is also an option.
<>:4: SyntaxWarning: "\K" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\K"? A raw string is also an option.
C:\Users\user\AppData\Local\Temp\ipykernel_5768\2201111097.py:3: SyntaxWarning: "\K" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\K"? A raw string is also an option.
  "E:\Kuliah\Sun Paper Source\Efficiency\BB_Maret_PM16.xlsx",
C:\Users\user\AppData\Local\Temp\ipykernel_5768\2201111097.py:4: SyntaxWarning: "\K" is an invalid escape sequence

,Date,GSM,Total NBKP,Total LBKP,Total BB Recycle,Sub Total Pulp+Broke,% NBKP,% LBKP,% BB Recycle,% Pulp+Broke
57,2026-04-27 00:00:00,13/12,7125.0,0,0,45125.0,15.789474,0,0,1373337.642
58,2026-04-28 00:00:00,14/13/15,4687.5,0,0,29687.5,15.789474,0,0,1403025.142
59,2026-04-29 00:00:00,15/16,7312.5,0,0,46312.5,15.789474,0,0,1449337.642
60,2026-04-30 00:00:00,12/14/18/19,7125.0,0,0,45125.0,15.789474,0,0,1494462.642
61,0,0,0.0,0,0,0.0,0.000000,0,0,0.000


df_BB = pd.read_excel("../Efficiency/BB_Maret_PM16.xlsx", engine='openpyxl')
df_BB.drop(columns=['Grade'], inplace=True)
df_BB.head()

### Last Join

In [28]:
# Standarisasi kolom "date" ke bentuk datetime
df_joined['Date'] = pd.to_datetime(df_joined['Date'])
df_BB['Date'] = pd.to_datetime(df_BB['Date'])

In [29]:
# Joining
df_final = pd.merge(df_joined, df_BB, on='Date', how='inner') #inner
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 309 entries, 0 to 308
Data columns (total 38 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   Start_Time               309 non-null    datetime64[us]
 1   End_Time                 309 non-null    datetime64[us]
 2   Data_Count               309 non-null    int64         
 3   Mean_Creping             309 non-null    float64       
 4   Mean_Yankee Speed        309 non-null    float64       
 5   Mean_Pope Reel Speed     309 non-null    float64       
 6   Mean_Yankee Pressure     309 non-null    float64       
 7   Mean_Stock Flow          309 non-null    float64       
 8   Mean_Stock Consistency   309 non-null    float64       
 9   Mean_Flow Coating        309 non-null    float64       
 10  Mean_Flow Release        309 non-null    float64       
 11  Mean_Jet Wire Ratio      309 non-null    float64       
 12  Mean_Load KWH Refiner    309 non-null    float6

In [30]:
# Membaca file sebelumnya
df = df_final.copy()

# Cleaning - GSM
df['GSM'] = df['GSM'].astype(str)
df = df[~df['GSM'].str.contains('/', na=False)].copy()
df['GSM'] = df['GSM'].str.replace(',', '.')
df['GSM'] = df['GSM'].astype(float)
# Cleaning - MDWT
df['MDWT'] = pd.to_numeric(df['MDWT'], errors='coerce')
df = df.dropna(subset=['MDWT']).copy()

In [31]:
df['GSM'].tail(50)

205    13.0
206    13.0
207    13.0
208    13.0
209    13.0
210    13.0
211    13.0
212    13.0
213    13.0
214    13.0
215    13.0
216    13.0
217    13.0
218    13.0
219    13.0
220    13.0
221    13.0
222    13.0
223    13.0
224    13.0
225    13.0
226    13.0
227    13.0
228    13.0
229    13.0
230    13.0
231    13.0
232    13.0
233    13.0
234    13.0
235    13.0
236    13.0
237    13.0
238    13.0
239    13.0
240    13.0
241    13.0
242    13.0
243    13.0
244    13.0
245    13.0
246    13.0
247    13.0
248    13.0
249    13.0
250    13.0
251    13.0
252    13.0
253    13.0
254    13.0
Name: GSM, dtype: float64

In [32]:
df.info()

<class 'pandas.DataFrame'>
Index: 175 entries, 0 to 254
Data columns (total 38 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   Start_Time               175 non-null    datetime64[us]
 1   End_Time                 175 non-null    datetime64[us]
 2   Data_Count               175 non-null    int64         
 3   Mean_Creping             175 non-null    float64       
 4   Mean_Yankee Speed        175 non-null    float64       
 5   Mean_Pope Reel Speed     175 non-null    float64       
 6   Mean_Yankee Pressure     175 non-null    float64       
 7   Mean_Stock Flow          175 non-null    float64       
 8   Mean_Stock Consistency   175 non-null    float64       
 9   Mean_Flow Coating        175 non-null    float64       
 10  Mean_Flow Release        175 non-null    float64       
 11  Mean_Jet Wire Ratio      175 non-null    float64       
 12  Mean_Load KWH Refiner    175 non-null    float64    

### Convert Final Data to Excel

#### Group by Grade

In [33]:
# Memisahkan data berdasarkan awalan pada kolom Grade
df_toilet = df[df['Grade'].str.startswith('T', na=False) & 
               ~df['Grade'].str.startswith('TW', na=False)]
df_towel = df[df['Grade'].str.startswith('TW', na=False)]
df_facial = df[df['Grade'].str.startswith('FC', na=False)]

# Menampilkan jumlah data
print("Jumlah data Toilet :", len(df_toilet))
print("Jumlah data Towel  :", len(df_towel))
print("Jumlah data Facial :", len(df_facial))

Jumlah data Toilet : 0
Jumlah data Towel  : 0
Jumlah data Facial : 174


Simpan ke file Excel terpisah

In [34]:
df.to_excel('Final_PM16.xlsx', index=False)

In [35]:
#df_toilet.to_excel("Final_PM16-Toilet.xlsx", index=False)
#df_towel.to_excel("Final_PM16-Towel.xlsx", index=False)
df_facial.to_excel("Final_PM16-Facial.xlsx", index=False)